In [32]:
import os
import pandas as pd
import numpy as np
print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")

Libraries imported successfully!
Pandas version: 2.3.3


## Load Existing Clean CSV

In [33]:
csv_path = 'data/csv/clean_tunisair_flights.csv'
assert os.path.exists(csv_path), f'File not found: {csv_path}'
df_clean = pd.read_csv(csv_path, encoding='utf-8')
print(f"Loaded clean dataset: {len(df_clean)} records, {len(df_clean.columns)} columns")
df_clean.head(3)

Loaded clean dataset: 1148 records, 27 columns


,flight_date,flight_status,departure_airport,departure_iata,departure_delay_minutes,departure_scheduled,departure_estimated,departure_actual,arrival_airport,arrival_iata,...,flight_number,flight_iata,flight_icao,year,month,day,day_of_week,total_delay_minutes,route,is_delayed
0,2025-12-23 00:00:00,scheduled,Carthage,TUN,0.0,2025-12-23,2025-12-23,Unknown,King Abdulaziz International,JED,...,913,TU913,TAR913,2025.0,12.0,23.0,Tuesday,0.0,TUN → JED,0.0
1,2025-12-23 00:00:00,scheduled,Orly,ORY,0.0,2025-12-23,2025-12-23,Unknown,Djerba-Zarzis,DJE,...,635,TU635,TAR635,2025.0,12.0,23.0,Tuesday,0.0,ORY → DJE,0.0
2,2025-12-23 00:00:00,scheduled,Carthage,TUN,0.0,2025-12-23,2025-12-23,Unknown,King Abdulaziz International,JED,...,713,TU713,TAR713,2025.0,12.0,23.0,Tuesday,0.0,TUN → JED,0.0


## Normalize flight_date to yyyy-mm-dd (preserve dd/mm/yyyy)

In [34]:
# Robustly convert flight_date strings to yyyy-mm-dd without dropping dd/mm/yyyy
import re

if 'flight_date' in df_clean.columns:
    s = df_clean['flight_date'].astype(str).str.strip()
    # Detect dd/mm/yyyy or dd-mm-yyyy patterns and parse with dayfirst=True
    ddmmyyyy_slash = s.str.match(r'^\d{1,2}/\d{1,2}/\d{4}$')
    ddmmyyyy_dash  = s.str.match(r'^\d{1,2}-\d{1,2}-\d{4}$')

    if ddmmyyyy_slash.any():
        parsed_dd = pd.to_datetime(s[ddmmyyyy_slash], dayfirst=True, errors='coerce')
        s.loc[ddmmyyyy_slash] = parsed_dd.dt.strftime('%Y-%m-%d')
    if ddmmyyyy_dash.any():
        parsed_dd = pd.to_datetime(s[ddmmyyyy_dash], dayfirst=True, errors='coerce')
        s.loc[ddmmyyyy_dash] = parsed_dd.dt.strftime('%Y-%m-%d')

    # For remaining values: general parse; keep original on failure
    remaining = ~(ddmmyyyy_slash | ddmmyyyy_dash)
    if remaining.any():
        parsed_other = pd.to_datetime(s[remaining], errors='coerce', utc=True)
        parsed_other = parsed_other.dt.tz_localize(None)
        formatted_other = parsed_other.dt.strftime('%Y-%m-%d')
        s.loc[remaining] = formatted_other.where(parsed_other.notna(), s[remaining])

    df_clean['flight_date'] = s
    print('✅ flight_date normalized to yyyy-mm-dd where parsable; originals preserved on failure.')
else:
    print('Warning: flight_date column not found.')

✅ flight_date normalized to yyyy-mm-dd where parsable; originals preserved on failure.


## Convert numeric columns and handle missing values

In [35]:
# Numeric conversion for delays (if present)
for col in ['departure_delay_minutes', 'arrival_delay_minutes']:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Fill numeric NaN delays with 0
if 'departure_delay_minutes' in df_clean.columns:
    df_clean['departure_delay_minutes'].fillna(0, inplace=True)
if 'arrival_delay_minutes' in df_clean.columns:
    df_clean['arrival_delay_minutes'].fillna(0, inplace=True)

# Fill categorical NaNs with 'Unknown'
categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns
df_clean[categorical_cols] = df_clean[categorical_cols].fillna('Unknown')

print('✅ Numeric conversions applied and missing values handled.')

✅ Numeric conversions applied and missing values handled.


C:\Users\USER\AppData\Local\Temp\ipykernel_11940\3095943520.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['departure_delay_minutes'].fillna(0, inplace=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_11940\3095943520.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy

## Add derived columns

In [36]:
# Derived from flight_date and add route using " to "
if 'flight_date' in df_clean.columns:
    dt = pd.to_datetime(df_clean['flight_date'], errors='coerce')
    df_clean['year'] = dt.dt.year.astype('Int64')
    df_clean['month'] = dt.dt.month.astype('Int64')
    df_clean['day'] = dt.dt.day.astype('Int64')
    df_clean['day_of_week'] = dt.dt.day_name()

# Total delay, delayed flag
if {'departure_delay_minutes','arrival_delay_minutes'}.issubset(df_clean.columns):
    df_clean['total_delay_minutes'] = df_clean['departure_delay_minutes'] + df_clean['arrival_delay_minutes']
    df_clean['is_delayed'] = df_clean['total_delay_minutes'] > 15

# Route using " to " between IATA codes
if {'departure_iata','arrival_iata'}.issubset(df_clean.columns):
    df_clean['route'] = df_clean['departure_iata'].astype(str) + ' to ' + df_clean['arrival_iata'].astype(str)

# Normalize any existing route arrows to " to "
if 'route' in df_clean.columns:
    df_clean['route'] = df_clean['route'].astype(str).str.replace('→', ' to ', regex=False)

print('✅ Derived columns updated. Route now uses " to " delimiter.')
df_clean.head(3)

✅ Derived columns updated. Route now uses " to " delimiter.


,flight_date,flight_status,departure_airport,departure_iata,departure_delay_minutes,departure_scheduled,departure_estimated,departure_actual,arrival_airport,arrival_iata,...,flight_number,flight_iata,flight_icao,year,month,day,day_of_week,total_delay_minutes,route,is_delayed
0,2025-12-23,scheduled,Carthage,TUN,0.0,2025-12-23,2025-12-23,Unknown,King Abdulaziz International,JED,...,913,TU913,TAR913,2025,12,23,Tuesday,0.0,TUN to JED,False
1,2025-12-23,scheduled,Orly,ORY,0.0,2025-12-23,2025-12-23,Unknown,Djerba-Zarzis,DJE,...,635,TU635,TAR635,2025,12,23,Tuesday,0.0,ORY to DJE,False
2,2025-12-23,scheduled,Carthage,TUN,0.0,2025-12-23,2025-12-23,Unknown,King Abdulaziz International,JED,...,713,TU713,TAR713,2025,12,23,Tuesday,0.0,TUN to JED,False


## Data summary

In [37]:
print(f'Total Records: {len(df_clean)}')
print(f'Total Columns: {len(df_clean.columns)}')
print('Data Types:')
print(df_clean.dtypes.value_counts())
print('Numeric summary:')
display(df_clean.select_dtypes(include=['number']).describe())

if 'flight_status' in df_clean.columns:
    print('Flight Status Distribution:')
    print(df_clean['flight_status'].value_counts())

Total Records: 1148
Total Columns: 27
Data Types:
object     19
float64     3
Int64       3
int64       1
bool        1
Name: count, dtype: int64
Numeric summary:


,departure_delay_minutes,arrival_delay_minutes,flight_number,year,month,day,total_delay_minutes
count,1148.000000,1148.000000,1148.000000,1148.0,1148.0,1148.0,1148.000000
mean,23.155052,33.108885,671.697735,2024.364983,11.182927,22.309233,56.263937
std,45.362210,53.133719,651.303005,1.226016,1.68474,8.697935,91.805654
min,0.000000,0.000000,8.000000,2022.0,7.0,1.0,0.000000
25%,0.000000,0.000000,397.000000,2025.0,12.0,22.0,0.000000
50%,0.000000,6.500000,701.000000,2025.0,12.0,25.0,10.000000
75%,30.000000,49.000000,753.750000,2025.0,12.0,28.0,83.250000
max,585.000000,585.000000,9020.000000,2025.0,12.0,31.0,1170.000000


Flight Status Distribution:
flight_status
scheduled    428
landed       405
active       221
Unknown       88
unknown        2
diverted       2
cancelled      2
Name: count, dtype: int64


## Export enriched dataset

In [38]:
output_file = 'data/csv/clean_tunisair_flights_enriched.csv'
df_export = df_clean.copy()
# Convert nullable Int64 to object for CSV compatibility
for col in ['year', 'month', 'day', 'week_of_year', 'quarter']:
    if col in df_export.columns and str(df_export[col].dtype) == 'Int64':
        df_export[col] = df_export[col].astype('Int64').astype('object')

df_export.to_csv(output_file, index=False, encoding='utf-8')
print(f'✅ Enriched dataset exported to: {output_file}')
print(f'Records exported: {len(df_export)}')
print(f'Columns exported: {len(df_export.columns)}')

✅ Enriched dataset exported to: data/csv/clean_tunisair_flights_enriched.csv
Records exported: 1148
Columns exported: 27
